# Lista 5 — Zadanie 4: Klasyfikacja decoder-only LLM (Qwen) (20 pkt)

In [ ]:
import subprocess
import sys

PKGS = [
    "transformers",
    "datasets",
    "scikit-learn",
    "langchain-core",
    "langchain-huggingface",
    "accelerate",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PKGS], check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "torchaudio", "--upgrade"],
    check=False,
)
try:
    import torchaudio
except (ImportError, RuntimeError):
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "torchaudio"],
        capture_output=True,
    )

In [2]:
import sys
from pathlib import Path

TASK5_DIR = Path("..").resolve()
if str(TASK5_DIR) not in sys.path:
    sys.path.insert(0, str(TASK5_DIR))

import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate

from common.data import load_polemo_test
from common.labels import map_text_to_class
from common.metrics import evaluate_predictions, print_evaluation

## Krok 1: Konfiguracja

In [ ]:
LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
TEMPERATURE = 0.1
SAMPLE_SIZE = 100

print(f"GPU dostępne: {torch.cuda.is_available()}")

GPU dostępne: False


/home/dawid/Pulpit/artificial-intelligence-and-knowledge-engineering/task-5/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 804: forward compatibility was attempted on non supported HW (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


## Krok 2: Ładowanie danych

In [4]:
examples = load_polemo_test()
if SAMPLE_SIZE is not None:
    examples = examples[:SAMPLE_SIZE]

sentences = [ex["sentence"] for ex in examples]
y_true = [ex["class"] for ex in examples]
print(f"Liczba próbek: {len(sentences)}")

Liczba próbek: 100


## Krok 3: Ładowanie modelu LLM

In [5]:
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

hf_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=TEMPERATURE,
    do_sample=TEMPERATURE > 0,
    max_new_tokens=10,
    pad_token_id=tokenizer.eos_token_id,
    return_full_text=False,
)

llm = HuggingFacePipeline(pipeline=hf_pipeline)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

/home/dawid/Pulpit/artificial-intelligence-and-knowledge-engineering/task-5/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:1074: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'temperature', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


## Krok 4: Prompt i łańcuch LangChain

In [6]:
BASIC_PROMPT = """Classify the text sentiment into one of three classes: positive, negative, neutral.
Reply with only one word — the class name. Do not explain your choice.

Text: {text}
Class: """

prompt = PromptTemplate.from_template(BASIC_PROMPT)
llm_chain = prompt | llm

## Krok 5: Klasyfikacja z mapowaniem etykiet

In [ ]:
def classify_with_llm(chain, sentences):
    predictions = []
    raw_answers = []
    unmapped = []

    for sentence in tqdm(sentences, desc="Klasyfikacja LLM"):
        answer = chain.invoke({"text": sentence})
        raw_answers.append(answer)

        clean_answer = answer.strip().split("\n")[0].lower()
        mapped = map_text_to_class(clean_answer)
        if mapped is None:
            unmapped.append((sentence[:60], answer))
            mapped = "neutral"
        predictions.append(mapped)

    if unmapped:
        print(f"\nNiezmapowane odpowiedzi: {len(unmapped)}")
        for text, ans in unmapped[:3]:
            print(f"  Odpowiedź: '{ans}' | Tekst: {text}...")

    return predictions, raw_answers


y_pred, raw_answers = classify_with_llm(llm_chain, sentences)

Klasyfikacja LLM:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length

## Krok 6: Podgląd surowych odpowiedzi LLM

In [8]:
for i in range(min(5, len(sentences))):
    print(f"\nTekst:    {sentences[i][:100]}...")
    print(f"Prawda:   {y_true[i]}")
    print(f"LLM:      {raw_answers[i]}")
    print(f"Zmapowano: {y_pred[i]}")


Tekst:    Leczyła m się u niej parę lat i nic mi nie pomogła , a jak zmieniła m lekarza po krótkim czasie zoba...
Prawda:   minus
LLM:      Classify the text sentiment into one of three classes: positive, negative, neutral.
Reply with only one word — the class name.

Text: Leczyła m się u niej parę lat i nic mi nie pomogła , a jak zmieniła m lekarza po krótkim czasie zobaczyła m już poprawę a w tej chwili jestem już bez leków 6 lat i jest wszystko dobrze . Dr Ciborska leczyła mnie na depresję a potem przez dr Kopystecką miała m rozpoznany zespół maniakalno depresyjny i odtąd zmianę leków i przede wszystkim wysłuchała mnie z zaagażowaniem a nie jak dr Ciborska aby mnie zbyć
Class: Positive

Explanation:
The text expresses gratitude and satisfaction
Zmapowano: minus

Tekst:    była m u tego lekarza na badaniu usg - 3D ( które trochę kosztuje ) , jestem bardzo nie zadowolona z...
Prawda:   minus
LLM:      Classify the text sentiment into one of three classes: positive, negative, neutral.

## Krok 7: Ewaluacja

In [9]:
results = evaluate_predictions(y_true, y_pred)
print_evaluation(results, title=f"Qwen LLM baseline — {LLM_MODEL}")


Qwen LLM baseline — Qwen/Qwen2.5-1.5B-Instruct
Accuracy:    0.4600
F1 (macro):  0.2100
F1 (weighted): 0.2899

Raport per klasa:
              precision    recall  f1-score   support

       minus       0.46      1.00      0.63        46
     neutral       0.00      0.00      0.00        21
        plus       0.00      0.00      0.00        33

    accuracy                           0.46       100
   macro avg       0.15      0.33      0.21       100
weighted avg       0.21      0.46      0.29       100

Macierz pomyłek (wiersze = prawda, kolumny = predykcja):
Klasy: ['minus', 'neutral', 'plus']
[[46  0  0]
 [21  0  0]
 [33  0  0]]
